# 1D Wave Equation - Propagation and Reflection

This notebook demonstrates the derivation and numerical implementation of the **1D Linear Wave Equation**.

## 1. Governing Equation

The 1D second-order wave equation is:
$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}$$

Where:
- $u(x, t)$: Displacement pulse
- $c$: Wave speed

## 2. Detailed Analytical Trace (Sympy)

We explore both the **D'Alembert Solution** and **Separation of Variables**.

In [ ]:
import sympy as sp
from IPython.display import display, Math

x, t, c = sp.symbols('x t c', real=True, positive=True)

# 1. D'Alembert Verification
f = sp.Function('f')(x - c*t)
g = sp.Function('g')(x + c*t)
ansatz_da = f + g

wave_op = ansatz_da.diff(t, t) - c**2 * ansatz_da.diff(x, x)
display(Math(f"\\text{{Wave Operator on D'Alembert ansatz: }} {sp.simplify(wave_op)} = 0"))

# 2. Separation of Variables trace
X = sp.Function('X')(x)
Phi = sp.Function('Phi')(t)
lam = sp.symbols('lambda', real=True, positive=True)

ode_spatial = sp.Eq(X.diff(x, x) + lam**2 * X, 0)
ode_temporal = sp.Eq(Phi.diff(t, t) + (c*lam)**2 * Phi, 0)

sol_x = sp.dsolve(ode_spatial, X)
sol_phi = sp.dsolve(ode_temporal, Phi)

display(Math(f"\\text{{Spatial ODE: }} {sp.latex(ode_spatial)}"))
display(Math(f"\\text{{Temporal ODE (Oscillatory): }} {sp.latex(ode_temporal)}"))
display(Math(f"X(x) = {sp.latex(sol_x.rhs)}"))
display(Math(f"\\Phi(t) = {sp.latex(sol_phi.rhs)}"))

## 3. Numerical Implementation

We use a 2nd-order central difference for time and space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def solve_wave_numpy(nx, L, nt, dt, c, u_init):
    dx = L / (nx - 1)
    u = np.zeros((3, nx)) # [prev, curr, next]
    u[0, :] = u_init.copy()
    u[1, :] = u_init.copy()
    
    r_sq = (c * dt / dx)**2
    
    for _ in range(nt):
        u[2, 1:-1] = 2*u[1, 1:-1] - u[0, 1:-1] + r_sq * (u[1, 2:] - 2*u[1, 1:-1] + u[1, 0:-2])
        # Dirichlet BCs
        u[2, 0] = 0
        u[2, -1] = 0
        
        u[0, :] = u[1, :]
        u[1, :] = u[2, :]
    return u[1, :]

nx = 201
L = 2.0
c_val = 1.0
dt = 0.005
nt = 200
x_vals = np.linspace(0, L, nx)

u0 = np.exp(-200 * (x_vals - 1.0)**2)
u_final = solve_wave_numpy(nx, L, nt, dt, c_val, u0)

plt.figure(figsize=(10, 5))
plt.plot(x_vals, u0, 'k:', label='Initial condition')
plt.plot(x_vals, u_final, 'b-', label='Numerical Field')
plt.title(f"1D Wave Equation Propagation (t = {nt*dt:.2f}s)")
plt.legend()
plt.grid(True)
plt.show()